In [2]:
import os
import requests
from datetime import datetime
from langfuse import get_client, Evaluation
from qdrant_client.http.models import SearchParams
from langchain_qdrant import QdrantVectorStore, RetrievalMode

from polylex_chatbot.env import load_project_env

env_path = load_project_env()

from polylex_chatbot.config import EMBEDDING_MODEL_CONFIG, SPARSE_MODEL_CONFIG_FR, DB_DENSE_INDEX_CONFIG, DB_SPARSE_INDEX_CONFIG_FR

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [7]:
langfuse = get_client()
DATASET_NAME = "20250520_clean_dev_dataset"
dataset = langfuse.get_dataset(DATASET_NAME)
RUN_NAME = datetime.now().isoformat()

embeddings = EMBEDDING_MODEL_CONFIG

qdrant = QdrantVectorStore.from_existing_collection(
    url=os.getenv("QDRANT_URL"),
    embedding=embeddings,
    sparse_embedding=SPARSE_MODEL_CONFIG_FR,
    collection_name=os.getenv("DB_COLLECTION_NAME"),
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name=list(DB_DENSE_INDEX_CONFIG.keys())[0],
    sparse_vector_name=list(DB_SPARSE_INDEX_CONFIG_FR.keys())[0]
)

K = 100

In [9]:
# source : https://gitlab.epfl.ch/rcp/aiaas/client-usage-examples/-/blob/main/requests-reranker.py

# TODO : top_n doit etre une constante qq part
def rerank_documents(api_key, base_url, query, documents, model, top_n=20):
    """
    Reranks documents based on their relevance to a given query.

    Args:
        api_key (str): Your API key
        base_url (str): The base URL for the reranking API
        query (str): The query to rerank documents for
        documents (list[str]): A list of documents to rerank
        model (str): The model to use for reranking.
        top_n (int, optional): The number of top documents to return. Defaults to 3.

    Returns:
        list[str]: The reranked documents
    """
    url = f"{base_url}/rerank"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "query": query,
        "documents": documents,
        "top_n": top_n
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    result = response.json()

    ranked_documents = sorted(result["results"], key=lambda x: x["relevance_score"], reverse=True)

    return ranked_documents

In [10]:
def make_retrieval_task(reranker_model_name, reranker_api_key, base_url):
    def retrieval_task(*, item, **kwargs):
        query = item.input.get("query")
        hits = qdrant.similarity_search_with_score(query, k=K, search_params=SearchParams(exact=True))

        # reranker
        documents = [document.page_content for document, _ in hits]
        reranked_hits = rerank_documents(reranker_api_key, base_url, query, documents, reranker_model_name)

        retrieved_doc_ids = []
        reranked_scores = []

        for hit in reranked_hits:
            original_index = hit["index"]
            rerank_score = hit["relevance_score"]
            document, _ = hits[original_index]
            doc_id = document.metadata.get("doc_id")
            retrieved_doc_ids.append(doc_id)
            reranked_scores.append(rerank_score)

        return {
            "retrieved_doc_ids": retrieved_doc_ids,
            "retrieved_scores": reranked_scores
        }
    return retrieval_task

In [11]:
def make_hit_at_x_evaluator(x):
    def hit_at_x_evaluator(*, output, metadata, **kwargs):
        expected_doc_id = metadata.get("expected_doc_id")
        retrieved_doc_ids_top_k = output.get("retrieved_doc_ids")[:x]
        retrieved_scores_top_k = output.get("retrieved_scores")[:x]
        value = 1.0 if expected_doc_id in retrieved_doc_ids_top_k else 0.0
        return Evaluation(
            name=f"hit_at_{x}",
            value=value,
            comment=f"expected_doc_id={expected_doc_id}, top{x}={retrieved_doc_ids_top_k} with score {retrieved_scores_top_k}"
        )
    return hit_at_x_evaluator

def mrr_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == expected_doc_id:
            reciprocal_rank = 1.0 / rank
            break
    return Evaluation(
        name="mrr_doc",
        value=reciprocal_rank,
        comment=f"expected_doc_id={expected_doc_id} and retrieved={retrieved_doc_ids}",
    )

def nb_correct_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    nb_correct_docs = 0
    for retrieved_doc_id in retrieved_doc_ids:
        if retrieved_doc_id == expected_doc_id:
            nb_correct_docs += 1
    return Evaluation(
        name="nb_correct_doc",
        value=nb_correct_docs,
        comment=f"expected_doc_id={expected_doc_id} and retrieved={retrieved_doc_ids}",
    )

In [12]:
result = dataset.run_experiment(
    name=os.getenv("DB_COLLECTION_NAME"),
    run_name=RUN_NAME,
    description=f"Reranking experiment to score retrieval task on {DATASET_NAME} dataset using {os.getenv("DB_COLLECTION_NAME")} collection",
    task=make_retrieval_task(os.getenv("MODEL_RERANKER_NAME"), os.getenv("MODEL_RERANKER_API_KEY"), os.getenv("MODELS_BASE_URL")),
    evaluators=[
        make_hit_at_x_evaluator(1),
        make_hit_at_x_evaluator(2),
        make_hit_at_x_evaluator(3),
        make_hit_at_x_evaluator(4),
        make_hit_at_x_evaluator(5),
        make_hit_at_x_evaluator(10),
        make_hit_at_x_evaluator(15),
        make_hit_at_x_evaluator(20),
        mrr_doc_evaluator,
        nb_correct_doc_evaluator
    ],
    metadata={
        "collection_name": os.getenv("DB_COLLECTION_NAME"),
        "top_k": K,
    }
)

print(result.format())

Individual Results: Hidden (13 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: 20260616_164451_collection
📋 Run name: 2026-06-17T10:09:54.982964 - Reranking experiment to score retrieval task on 20250520_clean_dev_dataset dataset using 20260616_164451_collection collection\n13 items\nEvaluations:\n  • hit_at_5\n  • mrr_doc\n  • hit_at_1\n  • hit_at_3\n  • nb_correct_doc\n  • hit_at_10\n  • hit_at_4\n  • hit_at_15\n  • hit_at_20\n  • hit_at_2\n\nAverage Scores:\n  • hit_at_5: 0.692\n  • mrr_doc: 0.646\n  • hit_at_1: 0.615\n  • hit_at_3: 0.615\n  • nb_correct_doc: 2.923\n  • hit_at_10: 0.769\n  • hit_at_4: 0.692\n  • hit_at_15: 0.769\n  • hit_at_20: 0.769\n  • hit_at_2: 0.615\n\n🔗 Dataset Run:\n   http://localhost:3000/project/cmmvqmjqn0007nw07hbqsche1/datasets/cmpdor8yr0026nw07btz59oao/runs/38d635f6-4a34-4cf1-998f-bc45cbdae944
